In [1]:
%load_ext autoreload
%autoreload 2

from library import *
from SIDER_dataset.libraries.XofN_library import *
import itertools

In [2]:
# Test generate_feature_rankings
rows = 1000
features = 10
labels = 1
random_state = 42
ranking_criterion = "AUROC_MDI_MDA"
(
    test_df,
    label_columns,
    feature_columns,
    binary_feature_columns,
    prob_feature_columns,
) = get_test_df(rows, features, labels, random_state)

learn_set, val_set = train_test_split(test_df, test_size=0.2, random_state=random_state)
rankings = generate_feature_rankings(
    learn_set, val_set, label_columns[0], ranking_criterion
)

print(rankings["MDA"].sum())

rankings

TypeError: generate_feature_rankings() takes 2 positional arguments but 4 were given

In [ ]:
# Test generate groupings
rows = 1000
features = 1000
labels = 1
random_state = 42
ranking_criterion = "MDI"
training_algorithm = "DT"
eval_criterion = "roc_auc"
max_size = 5
label = label_columns[0]
dataset_name = "test_df"
include_original_features = False
(
    test_df,
    label_columns,
    feature_columns,
    binary_feature_columns,
    prob_feature_columns,
) = get_test_df(rows, features, labels, random_state)
test_df = test_df[binary_feature_columns + label_columns]
parameters_name = "_".join(
    [
        dataset_name,
        label,
        ranking_criterion,
        training_algorithm,
        eval_criterion,
        str(max_size),
        "with_org" if include_original_features else "no_org",
    ]
)
print(f"Running with parameters: {parameters_name}")

logging_path = f"XofN_datasets/auto_groupings/{parameters_name}_logs.txt"
print(f"Logs can be found in {logging_path}.")
logger = get_logger(logging_path)
logger.info(parameters_name)

learn_set, val_set = train_test_split(test_df, test_size=0.2, random_state=random_state)
rankings = generate_feature_rankings(learn_set, val_set, label, ranking_criterion)

print(f"Generating groupings...")

XofN_groupings, results = generate_XofN_list(
    training_algorithm,
    learn_set,
    val_set,
    rankings,
    eval_criterion,
    max_size,
    label,
    logger,
)

for handler in logging.root.handlers[:]:
    handler.close()
    logging.root.removeHandler(handler)

Running with parameters: test_df_label_1_MDI_DT_roc_auc_5_no_org
Logs can be found in XofN_datasets/auto_groupings/test_df_label_1_MDI_DT_roc_auc_5_no_org_logs.txt.

generate_feature_rankings -> Generating rankings based on ranking_criterion:MDI.


c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoro


train
 Training with: algorithm: DT    instances: 1000   features: 500   labels: 1 ['label_1']
 Training...
Generating groupings...


TypeError: generate_XofN_list() takes 8 positional arguments but 9 were given

In [81]:
# Test group_features
rows = 1000
features = 20
labels = 1
random_state = 42
ranking_criterion = "MDI"
(
    dataset,
    label_columns,
    feature_columns,
    binary_feature_columns,
    prob_feature_columns,
) = get_test_df(rows, features, labels, random_state)

groups = [
    ["b_feature_1"],
    ["b_feature_8", "b_feature_9", "b_feature_10"],
    ["b_feature_3", "b_feature_4", "b_feature_5"],
]

dataset = group_features(dataset, label_columns, groups, True)
dataset


group_features
Number of original features: 20
Number of generated XofN group features: 3
Number of features in XofN groups: 7
Number of features not in XofN groups: 3
Number of XofN group features with a single item: 1


,b_feature_1,b_feature_2,b_feature_3,b_feature_4,b_feature_5,b_feature_6,b_feature_7,b_feature_8,b_feature_9,b_feature_10,...,p_feature_4,p_feature_5,p_feature_6,p_feature_7,p_feature_8,p_feature_9,p_feature_10,f_b_feature_8_b_feature_9_b_feature_10,f_b_feature_3_b_feature_4_b_feature_5,label_1
0,1,1,0,0,0,1,0,0,0,1,...,479,559,18,773,83,154,564,1,0,1
1,1,0,0,0,1,0,1,1,1,0,...,646,442,68,522,142,191,386,2,1,1
2,1,0,1,1,1,1,1,1,1,1,...,692,292,8,270,386,92,965,3,3,0
3,1,0,1,1,1,0,1,0,0,0,...,839,148,888,266,421,839,617,0,3,0
4,1,0,1,1,1,1,1,0,1,1,...,630,525,502,13,777,651,805,2,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1,1,0,0,0,1,0,0,0,1,...,479,559,18,773,83,154,564,1,0,1
996,1,1,0,0,0,1,0,0,0,1,...,479,559,18,773,83,154,564,1,0,1
997,1,1,0,0,0,1,0,0,0,1,...,479,559,18,773,83,154,564,1,0,1
998,1,1,0,0,0,1,0,0,0,1,...,479,559,18,773,83,154,564,1,0,1


In [ ]:
# Single label - No CV
rows = 1000
features = 100
labels = 6
random_state = 42
model_type = "RF"
df, label_columns, feature_columns, binary_feature_columns, prob_feature_columns = (
    get_test_df(rows, features, labels, random_state)
)

feature_threshold = round(len(df) * 1 / 100)

df, label_columns, feature_columns, binary_feature_columns, prob_feature_columns = (
    drop_uninformative_features(
        df,
        feature_threshold,
        label_columns,
        binary_feature_columns,
        prob_feature_columns,
    )
)

df = remove_inconsistent_duplicates(df, feature_columns, label_columns)

train_dataset, test_dataset = train_test_split(
    df,
    test_size=0.2,
    random_state=random_state,
)

model, importances = train(
    model_type,
    train_dataset,
    [label_columns[1]],
)

performance = test(model, test_dataset, [label_columns[1]])
performance


drop_uninformative_features
 Removing 0 uninformative binary features. Remaining binary features: 50
 Removing 0 uninformative probability scores features. Remaining probability features: 50
 Removing 2 uninformative features where all values are the same. Remaining features: 98

remove_inconsistent_duplicates
  Dropping 11 duplicate rows (keep first). Remaining rows: 989
  ❌ Deleting 10 inconsistent rows (same values for features but different for labels). Remaining: 979

train
 Training with:
    algorithm: RF
    instances: 783
    features: 103
    labels: 1 ['label_2']
 Training...


,accuracy,precision,recall,f1,roc_auc
label_2,0.52551,0.537634,0.5,0.518135,0.510469


In [ ]:
# Single label - CV
k = 10
model_type = "RF"
rows = 1000
features = 100
labels = 6
random_state = 42
df, label_columns, feature_columns, binary_feature_columns, prob_feature_columns = (
    get_test_df(rows, features, labels, random_state)
)

feature_threshold = round(len(df) * 1 / 100)

df, label_columns, feature_columns, binary_feature_columns, prob_feature_columns = (
    drop_uninformative_features(
        df,
        feature_threshold,
        label_columns,
        binary_feature_columns,
        prob_feature_columns,
    )
)

df = remove_inconsistent_duplicates(df, feature_columns, label_columns)

kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
performance_dfs = []

for fold, (train_idx, test_idx) in enumerate(kf.split(df)):
    print(f"Fold {fold + 1}/{k}")

    train_dataset = df.iloc[train_idx]
    test_dataset = df.iloc[test_idx]

    model, importances = train(model_type, train_dataset, [label_columns[1]])
    performance_df = test(model, test_dataset, [label_columns[1]])
    performance_df["fold"] = fold + 1
    performance_dfs.append(performance_df)
    print(performance_df.to_string())

all_results, mean_results = get_cv_results(performance_dfs)
all_results


drop_uninformative_features
 Removing 0 uninformative binary features. Remaining binary features: 50
 Removing 0 uninformative probability scores features. Remaining probability features: 50
 Removing 2 uninformative features where all values are the same. Remaining features: 98

remove_inconsistent_duplicates
  Dropping 11 duplicate rows (keep first). Remaining rows: 989
  ❌ Deleting 10 inconsistent rows (same values for features but different for labels). Remaining: 979
Fold 1/10

train
 Training with:
    algorithm: RF
    instances: 881
    features: 103
    labels: 1 ['label_2']
 Training...

test
 Testing with:
    instances: 98
         accuracy  precision    recall        f1   roc_auc  fold
label_2  0.602041      0.625  0.510204  0.561798  0.613911     1
Fold 2/10

train
 Training with:
    algorithm: RF
    instances: 881
    features: 103
    labels: 1 ['label_2']
 Training...

test
 Testing with:
    instances: 98
         accuracy  precision    recall        f1   roc_auc  

,index,accuracy,precision,recall,f1,roc_auc,fold
0,label_2,0.602041,0.625000,0.510204,0.561798,0.613911,1
1,label_2,0.551020,0.577778,0.509804,0.541667,0.488736,2
2,label_2,0.540816,0.777778,0.350000,0.482759,0.559649,3
3,label_2,0.561224,0.560976,0.479167,0.516854,0.577500,4
4,label_2,0.530612,0.526316,0.416667,0.465116,0.540625,5
5,label_2,0.551020,0.400000,0.444444,0.421053,0.484767,6
6,label_2,0.571429,0.600000,0.428571,0.500000,0.595377,7
7,label_2,0.530612,0.489362,0.511111,0.500000,0.545283,8
8,label_2,0.469388,0.437500,0.291667,0.350000,0.369583,9
9,label_2,0.536082,0.488889,0.500000,0.494382,0.536664,10


In [ ]:
# Multi label - CV
k = 10
model_type = "RF"
rows = 1000
features = 100
labels = 6
random_state = 42
(
    test_df,
    label_columns,
    feature_columns,
    binary_feature_columns,
    prob_feature_columns,
) = get_test_df(rows, features, labels, random_state)
feature_threshold = round(len(df) * 1 / 100)

all_labels_cv_results = []
mean_labels_results = []
for current_label in label_columns:
    print(f"\n--- Processing label: {current_label} ---")
    current_df = test_df.drop(
        columns=[label for label in label_columns if label != current_label]
    )

    (
        current_df,
        current_labels,
        current_features,
        current_binary_features,
        current_prob_features,
    ) = drop_uninformative_features(
        current_df,
        feature_threshold,
        [current_label],
        binary_feature_columns,
        prob_feature_columns,
    )

    current_df = remove_inconsistent_duplicates(
        current_df, current_features, current_labels
    )

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    performance_dfs = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df)):
        print(f"Fold {fold + 1}/{k}")

        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        model, importances = train(model_type, train_dataset, current_labels)
        performance_df = test(model, test_dataset, current_labels)
        performance_df["fold"] = fold + 1
        performance_dfs.append(performance_df)
        print(performance_df.to_string())

    all_cv_results, mean_results = get_cv_results(performance_dfs)
    print(mean_results.to_string())

    all_labels_cv_results.append(all_cv_results)
    mean_labels_results.append(mean_results)

all_labels_results_df = pd.concat(all_labels_cv_results).reset_index()
mean_labels_results_df = get_macro_results(mean_labels_results)
mean_labels_results_df


--- Processing label: label_1 ---

drop_uninformative_features
 Removing 0 uninformative binary features. Remaining binary features: 50
 Removing 0 uninformative probability scores features. Remaining probability features: 50
 Removing 2 uninformative features where all values are the same. Remaining features: 103

remove_inconsistent_duplicates
  Dropping 11 duplicate rows (keep first). Remaining rows: 989
  ❌ Deleting 4 inconsistent rows (same values for features but different for labels). Remaining: 985
Fold 1/10

train
 Training with: algorithm: RF    instances: 886   features: 103   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 99
         accuracy  precision    recall        f1   roc_auc  fold
label_1  0.474747   0.507463  0.641509  0.566667  0.435193     1
Fold 2/10

train
 Training with: algorithm: RF    instances: 886   features: 103   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 99
         accuracy  precision  recall        f1   

,label,accuracy,precision,recall,f1,roc_auc
0,label_1,0.510781,0.530643,0.744483,0.616434,0.481287
1,label_2,0.519666,0.513030,0.421691,0.456111,0.537996
2,label_3,0.491363,0.506827,0.567463,0.532084,0.486073
3,label_4,0.498495,0.509674,0.603446,0.551707,0.494412
4,label_5,0.533890,0.539717,0.494782,0.507963,0.559152
5,label_6,0.530066,0.495895,0.231526,0.309284,0.506297
6,Macro,0.514043,0.515964,0.510565,0.495597,0.510870


In [ ]:
k = 10
model_type = "RF"
rows = 1000
features = 10
labels = 6
random_state = 42
(
    test_df,
    label_columns,
    feature_columns,
    binary_feature_columns,
    prob_feature_columns,
) = get_test_df(rows, features, labels, random_state)
feature_threshold = round(len(test_df) * 1 / 100)
dataset_name = "test_dataset"
params = {
    "label": label_columns[:1],
    "ranking_criterion": ["roc_auc", "MDI"],
    "training_algorithm": ["DT"],
    "eval_criterion": ["roc_auc"],
    "include_original_features": [False, True],
    "max_size": [5],
}
all_labels_cv_results = []
mean_labels_results = []
for combo in itertools.product(*params.values()):
    config = dict(zip(params.keys(), combo))
    current_label = config["label"]
    ranking_criterion = config["ranking_criterion"]
    training_algorithm = config["training_algorithm"]
    eval_criterion = config["eval_criterion"]
    include_original_features = config["include_original_features"]
    max_size = config["max_size"]
    run_config = f"\n--- Running with label:'{current_label}' ranking_criterion:'{ranking_criterion}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criterion}' include_original_features:'{include_original_features}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            current_label,
            ranking_criterion,
            training_algorithm,
            eval_criterion,
            "with_org" if include_original_features else "no_org",
            str(max_size),
        ]
    )
    logging_path = f"XofN_datasets/auto_groupings/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    current_df = test_df.drop(
        columns=[label for label in label_columns if label != current_label]
    )
    # current_df = current_df.drop(columns=["CID"])

    (
        current_df,
        current_labels,
        current_features,
        current_binary_features,
        current_prob_features,
    ) = drop_uninformative_features(
        current_df,
        feature_threshold,
        [current_label],
        binary_feature_columns,
    )

    current_df = remove_inconsistent_duplicates(
        current_df, current_features, current_labels
    )

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    performance_dfs = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df)):
        print(f"\nFold {fold + 1}/{k}")

        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]
        cache_results_path = f"XofN_datasets/auto_groupings/{dataset_name}_{current_label}_cache_results.json"
        cache_results = get_cache_results(cache_results_path)

        learn_set, val_set = train_test_split(
            train_dataset, test_size=0.2, random_state=42
        )

        rankings = generate_feature_rankings(
            learn_set, val_set, current_label, ranking_criterion
        )

        XofN_groupings, results = generate_XofN_list(
            training_algorithm,
            learn_set,
            val_set,
            rankings,
            eval_criterion,
            max_size,
            current_label,
            cache_results,
            logger,
        )

        train_dataset = group_features(
            train_dataset,
            [current_label],
            XofN_groupings,
            include_original_features,
        )

        test_dataset = group_features(
            test_dataset,
            [current_label],
            XofN_groupings,
            include_original_features,
        )

        with open(cache_results_path, "w") as f:
            json.dump(cache_results, f, indent=4)

        model, importances = train(model_type, train_dataset, current_labels)
        performance_df = test(model, test_dataset, current_labels)
        performance_df["fold"] = fold + 1
        performance_df["ranking_criterion"] = ranking_criterion
        performance_df["training_algorithm"] = training_algorithm
        performance_df["eval_criterion"] = eval_criterion
        performance_df["include_original_features"] = (
            "with_org" if include_original_features else "no_org"
        )
        performance_df["max_size"] = str(max_size)
        performance_dfs.append(performance_df)
        print(performance_df.to_string())

    all_cv_results, mean_results = get_cv_results(performance_dfs)
    print(f"\nCV results: ", mean_results.to_string())

    all_labels_cv_results.append(all_cv_results)
    mean_labels_results.append(mean_results)

for handler in logging.root.handlers[:]:
    handler.close()
    logging.root.removeHandler(handler)

all_labels_cv_results_df = pd.concat(all_labels_cv_results).reset_index()
all_labels_mean_cv_results_df = get_mean_cv_results(all_labels_cv_results_df)
all_labels_mean_cv_results_df
# mean_labels_results_df = get_macro_results(all_labels_cv_results_df)
# mean_labels_results_df


--- Running with label:'label_1' ranking_criterion:'roc_auc' training_algorithm:'DT' eval_criterion:'roc_auc' include_original_features:'False' max_size:'5' ---
Logs can be found in XofN_datasets/auto_groupings/label_1_roc_auc_DT_roc_auc_no_org_5_logs.txt.

drop_uninformative_features
 Removing 0 uninformative binary features. Remaining binary features: 5
 Removing 2 uninformative features where all values are the same. Remaining features: 8

remove_inconsistent_duplicates
  Dropping 19 duplicate rows (keep first). Remaining rows: 981
  ❌ Deleting 2 inconsistent rows (same values for features but different for labels). Remaining: 979

Fold 1/10

generate_feature_rankings -> Generating rankings based on ranking_criterion:roc_auc.

train
 Training with: algorithm: DT    instances: 704   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1  roc_auc
b_feature_2  0.536723        0.0     0.0  0.0      0.5

train


c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoro


test
 Testing with:  instances: 98
         accuracy  precision    recall        f1   roc_auc  fold ranking_criterion training_algorithm eval_criterion include_original_features max_size
label_1   0.44898   0.414634  0.361702  0.386364  0.460576     1           roc_auc                 DT        roc_auc                    no_org        5

Fold 2/10

generate_feature_rankings -> Generating rankings based on ranking_criterion:roc_auc.

train
 Training with: algorithm: DT    instances: 704   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1  roc_auc
b_feature_2  0.564972        0.0     0.0  0.0      0.5

train
 Training with: algorithm: DT    instances: 704   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1   roc_auc
b_feature_3  0.564972        0.0     0.0  0.0  0.417273

train
 Training with: algorithm: DT    instances: 704 

c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoro


test
 Testing with:  instances: 98
         accuracy  precision    recall   f1   roc_auc  fold ranking_criterion training_algorithm eval_criterion include_original_features max_size
label_1  0.479592        0.5  0.333333  0.4  0.468085     2           roc_auc                 DT        roc_auc                    no_org        5

Fold 3/10

generate_feature_rankings -> Generating rankings based on ranking_criterion:roc_auc.

train
 Training with: algorithm: DT    instances: 704   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1  roc_auc
b_feature_2  0.531073        0.0     0.0  0.0      0.5

train
 Training with: algorithm: DT    instances: 704   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1   roc_auc
b_feature_3  0.531073        0.0     0.0  0.0  0.420405

train
 Training with: algorithm: DT    instances: 704   features

...

c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoro


test
 Testing with:  instances: 98
         accuracy  precision    recall        f1   roc_auc  fold ranking_criterion training_algorithm eval_criterion include_original_features max_size
label_1  0.428571   0.290323  0.209302  0.243243  0.458562     9               MDI                 DT        roc_auc                  with_org        5

Fold 10/10

generate_feature_rankings -> Generating rankings based on ranking_criterion:MDI.

train
 Training with: algorithm: DT    instances: 705   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1  roc_auc
b_feature_2  0.559322        0.0     0.0  0.0      0.5

train
 Training with: algorithm: DT    instances: 705   features: 1   labels: 1 ['label_1']
 Training...

test
 Testing with:  instances: 177
             accuracy  precision  recall   f1   roc_auc
b_feature_3  0.559322        0.0     0.0  0.0  0.514957

train
 Training with: algorithm: DT    instances: 705   f

c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoror\anaconda3\envs\phd\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\KostasVoro


test
 Testing with:  instances: 97
         accuracy  precision    recall        f1   roc_auc  fold ranking_criterion training_algorithm eval_criterion include_original_features max_size
label_1   0.56701   0.578947  0.458333  0.511628  0.557185    10               MDI                 DT        roc_auc                  with_org        5

CV results:           accuracy  precision   recall        f1   roc_auc
label                                                    
label_1  0.469966   0.409737  0.34765  0.374791  0.449405


,index,ranking_criterion,training_algorithm,eval_criterion,include_original_features,accuracy,precision,recall,f1,roc_auc
0,label_1,MDI,DT,roc_auc,no_org,0.481138,0.430004,0.391475,0.408012,0.468535
1,label_1,MDI,DT,roc_auc,with_org,0.469966,0.409737,0.347650,0.374791,0.449405
2,label_1,roc_auc,DT,roc_auc,no_org,0.476005,0.422732,0.372365,0.394683,0.460451
3,label_1,roc_auc,DT,roc_auc,with_org,0.465853,0.403235,0.340100,0.367776,0.453478


In [1]:
# exhaustive search example - produced in about 1 minute
# create combinations
import itertools

# Example list
nums = range(1, 100)

all_combinations = [
    list(c)
    for r in range(2, 6)
    for c in itertools.combinations(nums, r)
]

print(len(all_combinations))

75449220


In [2]:
# executed in about 6 secs
x = 0
for comb in all_combinations:
    x = x + 1
print(x)

75449220


In [3]:
print_xofn_groups_stats(generate_XofN_list_1st_fold())

XofN_groups: [['f_20', 'f_771', 'f_390', 'f_299', 'f_645'], ['f_23', 'f_513', 'f_355', 'f_146', 'f_394'], ['f_346', 'f_706', 'f_613', 'f_592'], ['f_308', 'f_815', 'f_582', 'f_578', 'f_257'], ['f_366', 'f_413', 'f_286'], ['f_15', 'f_151', 'f_351', 'f_192', 'f_292'], ['f_451', 'f_343', 'f_416', 'f_656', 'f_430'], ['f_287', 'f_471', 'f_333', 'f_709', 'f_31'], ['f_672', 'f_35', 'f_441', 'f_752', 'f_446'], ['f_406', 'f_361', 'f_556', 'f_616'], ['f_639', 'f_436', 'f_178', 'f_420', 'f_462'], ['f_3', 'f_214', 'f_143', 'f_712', 'f_699'], ['f_37', 'f_839', 'f_819', 'f_1', 'f_708'], ['f_392', 'f_631', 'f_285', 'f_824', 'f_635'], ['f_380', 'f_47', 'f_643', 'f_683', 'f_607'], ['f_452', 'f_328', 'f_340', 'f_255', 'f_545'], ['f_11', 'f_284', 'f_365', 'f_375', 'f_297'], ['f_614', 'f_131', 'f_384', 'f_569', 'f_552'], ['f_571', 'f_118', 'f_696', 'f_181', 'f_339'], ['f_24', 'f_647', 'f_439', 'f_464', 'f_803'], ['f_440', 'f_670', 'f_549', 'f_437', 'f_356'], ['f_617', 'f_729', 'f_14', 'f_585', 'f_454'], ['